# Kodra AI Agent: Cloud/Colab Training Notebook

**Product:** Kodra AI Agent  
**Model:** Kodra GPT (`KodraGPT`)  
**Core:** Kodra Core  
**Tagline:** CODE • THINK • CREATE

This notebook trains **Kodra GPT** on a GPU using Google Colab (or any Jupyter environment with a CUDA GPU). It clones the repository, prepares an approved dataset, trains the tokenizer, initializes a model configuration, runs training with periodic checkpointing, supports resuming after an interruption, evaluates the result, and generates sample completions.

No credentials are embedded in this notebook. If you want to persist checkpoints to Google Drive, mount it yourself in Colab and pass that path as `CHECKPOINT_DIR` below.

In [ ]:
# 1. Clone the repository and enter the Kodra Core directory
!git clone git@github.com:ChaceEthan/Kodra-ai.git
%cd Kodra-ai/kodra-core

In [ ]:
# 2. Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# 3. Detect GPU and print info
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device name:', torch.cuda.get_device_name(0))
    print('Device count:', torch.cuda.device_count())
    print('Total memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print('No GPU detected - training will fall back to CPU (slow for anything above kodra-tiny).')

In [ ]:
# 4. (Optional) Mount Google Drive for persistent checkpoint storage.
# Uncomment if you want checkpoints to survive a Colab session restart.
# from google.colab import drive
# drive.mount('/content/drive')
# CHECKPOINT_DIR = '/content/drive/MyDrive/kodra_checkpoints'
CHECKPOINT_DIR = 'checkpoints'

In [ ]:
# 5. Prepare dataset: build a manifest from an approved/licensed source tree.
# Point SOURCE_DIR at a directory you have the rights to train on. The bundled
# `data/code` sample corpus is used by default so this cell always runs.
from datasets.corpus_pipeline import build_manifest, write_manifest

SOURCE_DIR = 'data/code'
manifest = build_manifest(SOURCE_DIR, seed=42, license='project-sample', source='kodra-sample-corpus')
write_manifest(manifest, 'data/manifest.json')
print(f'Discovered {manifest.num_files} files, {manifest.total_chars} chars, languages: {manifest.language_counts}')

In [ ]:
# 6. Train the tokenizer (Phase 1 char tokenizer, or Phase 2 BPE tokenizer)
from datasets.sample_code import SAMPLE_CODE_CORPUS
from tokenizer.char_tokenizer import CharTokenizer
from tokenizer.bpe_tokenizer import ByteLevelBPETokenizer

USE_BPE = False  # set True to train the Phase 2 byte-level BPE tokenizer instead

if USE_BPE:
    tokenizer = ByteLevelBPETokenizer(vocab_size=8000)
    tokenizer.train(SAMPLE_CODE_CORPUS)
    tokenizer.save('tokenizer/vocab_bpe.json')
else:
    tokenizer = CharTokenizer()
    tokenizer.train(SAMPLE_CODE_CORPUS)
    tokenizer.save('tokenizer/vocab.json')

print('Tokenizer type:', tokenizer.tokenizer_type, '| vocab size:', tokenizer.vocab_size)

In [ ]:
# 7. Select a Kodra model configuration
from configs.model_sizes import get_model_size, validate_model_config
from model.gpt_model import KodraGPT

MODEL_SIZE = 'kodra-tiny'  # one of: kodra-tiny, kodra-small, kodra-base, kodra-medium
spec = get_model_size(MODEL_SIZE)
model_cfg = spec.config
model_cfg.vocab_size = tokenizer.vocab_size
validate_model_config(model_cfg)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = KodraGPT(model_cfg).to(device)
param_count = sum(p.numel() for p in model.parameters())
print(f'{spec.display_name}: {param_count:,} parameters ({param_count/1e6:.2f}M) on {device}')
print(f'Previously trained in this repo: {spec.trained}')

In [ ]:
# 8. Build the dataloader and trainer
from configs.config import TrainingConfig
from datasets.dataset import create_dataloader
from training.trainer import Trainer
from training.utils import set_seed

set_seed(42)
train_cfg = TrainingConfig(batch_size=16, learning_rate=3e-4, max_epochs=10)
train_loader = create_dataloader(SAMPLE_CODE_CORPUS, tokenizer, model_cfg.context_length, train_cfg.batch_size)
trainer = Trainer(model, train_cfg, train_loader, device=device, tokenizer_type=tokenizer.tokenizer_type)

In [ ]:
# 9. Resume from a checkpoint if one exists
import os
latest_ckpt = os.path.join(CHECKPOINT_DIR, 'kodra_gpt_latest.pt')
if os.path.exists(latest_ckpt):
    trainer.load_checkpoint(latest_ckpt)
    print(f'Resumed from step {trainer.step_count}')
else:
    print('No existing checkpoint found - starting fresh.')

In [ ]:
# 10. Train, checkpointing periodically (every epoch here; adjust to your needs)
total_steps = train_cfg.max_epochs * len(train_loader)
for epoch in range(1, train_cfg.max_epochs + 1):
    avg_loss = trainer.train_epoch(epoch, total_steps=total_steps)
    val_loss = trainer.evaluate()  # None unless a val_loader was configured
    trainer.save_latest_and_best(CHECKPOINT_DIR, val_loss=val_loss)
    tps = trainer.history[-1]['tokens_per_sec'] if trainer.history else 0.0
    print(f'Epoch {epoch}/{train_cfg.max_epochs} | avg_loss={avg_loss:.4f} | step={trainer.step_count} | tokens/sec={tps:.0f}')

In [ ]:
# 11. Evaluate (language-model + code-completion + syntax)
from evaluation.evaluator import full_evaluation_report
import json as _json

report = full_evaluation_report(model, tokenizer, device, train_loader)
print(_json.dumps(report, indent=2, default=str))

In [ ]:
# 12. Generate a sample code completion with the trained model
from inference.generator import CodeGenerator

generator = CodeGenerator(model, tokenizer, device)
sample = generator.generate('def quicksort(arr):', max_new_tokens=64, temperature=0.7, top_k=40)
print(sample)